# Sipply — Fine-tune Qwen2.5-7B with LoRA

**Before running:** Runtime > Change runtime type > **T4 GPU**

This notebook fine-tunes Qwen2.5-7B-Instruct on article and cluster summarization tasks using LoRA. No manual uploads needed — training data is pulled from GitHub automatically.

In [ ]:
# Cell 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# Cell 2: Install dependencies
!pip install -q transformers==4.47.1 peft==0.13.2 trl==0.12.2 bitsandbytes==0.45.0 accelerate==1.2.1 datasets==3.2.0 rouge-score sentencepiece

In [ ]:
# Cell 3: Download training data from GitHub
GITHUB_RAW = 'https://raw.githubusercontent.com/fanyang-888/personal-ai-intelligence-tool/main/backend/data'
!wget -q {GITHUB_RAW}/finetune_train.jsonl -O finetune_train.jsonl
!wget -q {GITHUB_RAW}/finetune_eval.jsonl  -O finetune_eval.jsonl

import json
train_raw = [json.loads(l) for l in open('finetune_train.jsonl') if l.strip()]
eval_raw  = [json.loads(l) for l in open('finetune_eval.jsonl')  if l.strip()]
from collections import Counter
print(f'Train: {len(train_raw)}  Eval: {len(eval_raw)}')
print('Tasks:', dict(Counter(r['task'] for r in train_raw)))

In [ ]:
# Cell 4: Load tokenizer
from transformers import AutoTokenizer
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'
print('Tokenizer loaded')

In [ ]:
# Cell 5: Format dataset
from datasets import Dataset

def to_hf_dataset(rows):
    texts = [
        tokenizer.apply_chat_template(r['messages'], tokenize=False, add_generation_prompt=False)
        for r in rows
    ]
    return Dataset.from_dict({'text': texts})

train_ds = to_hf_dataset(train_raw)
eval_ds  = to_hf_dataset(eval_raw)
print('Sample:')
print(train_ds[0]['text'][:400], '...')

In [ ]:
# Cell 6: Load model in 4-bit
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.config.use_cache = False
print('Model loaded (4-bit)')

In [ ]:
# Cell 7: Apply LoRA
from peft import LoraConfig, get_peft_model, TaskType

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
))
model.print_trainable_parameters()

In [ ]:
# Cell 8: Train
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = './sipply-qwen-lora'

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    dataset_text_field='text',
    max_seq_length=2048,
    packing=False,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        per_device_eval_batch_size=2,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        bf16=True,
        logging_steps=20,
        eval_strategy='steps',
        eval_steps=100,
        save_strategy='steps',
        save_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        report_to='none',
        optim='paged_adamw_8bit',
    ),
)
print('Training started — ~45 min on T4...')
trainer.train()
print('Done!')

In [ ]:
# Cell 9: ROUGE evaluation
import numpy as np
from rouge_score import rouge_scorer as rs

scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
model.eval()
results = []

for row in eval_raw[:50]:
    prompt = tokenizer.apply_chat_template(
        row['messages'][:-1], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    pred = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    ref  = row['messages'][-1]['content']
    try:
        pred_text = json.loads(pred).get('short_summary') or json.loads(pred).get('summary', '')
        ref_text  = json.loads(ref).get('short_summary')  or json.loads(ref).get('summary', '')
    except Exception:
        pred_text, ref_text = pred[:400], ref[:400]
    results.append(scorer.score(ref_text, pred_text))

print('ROUGE scores (50 eval examples):')
for m in ['rouge1', 'rouge2', 'rougeL']:
    f1s = [r[m].fmeasure for r in results]
    print(f'  {m}: {np.mean(f1s):.4f} (+/-{np.std(f1s):.4f})')

In [ ]:
# Cell 10: Quick qualitative test
test = 'Title: OpenAI releases o3-mini\n\nArticle text:\nOpenAI released o3-mini, a reasoning model scoring 87.3% on AIME 2024 at $1.10/M input tokens, cheaper than o1.'
msgs = [{'role': 'system', 'content': eval_raw[0]['messages'][0]['content']},
        {'role': 'user',   'content': test}]
prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

In [ ]:
# Cell 11: Save and zip adapter for download
adapter_path = f'{OUTPUT_DIR}/final-adapter'
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

!zip -r sipply-adapter.zip {adapter_path}

from google.colab import files
files.download('sipply-adapter.zip')
print('Download started!')